#MOUNTING DRIVE

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#IMPORTING LIBRARIES

In [8]:
!pip install torch torchvision pandas numpy pillow opencv-python matplotlib
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
import torchvision
from torchvision import transforms as T
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import cv2
import torchvision.transforms.v2 as v2 # improves generalization and detection robustness.

DATA_DIR = "/content/drive/MyDrive/LR _25/train" # <-- edit if needed
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
ANNOTATIONS_FILE = os.path.join(DATA_DIR, '_annotations.csv')

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Device:', device)

Device: cpu


#CUSTOM DATASET FOR R-CNN


In [ ]:
class CraterDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, min_size=None):
        self.annotations = pd.read_csv(csv_file) #stores all rows from the annotations file

        # check required columns
        required = {'filename', 'xmin', 'ymin', 'xmax', 'ymax'}
        if not required.issubset(set(self.annotations.columns)):
            raise ValueError(f"CSV must contain columns: {required}")

        # these must always run
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = self.annotations['filename'].unique().tolist()
        self.min_size = min_size

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        filename = self.img_names[idx]
        img_path = os.path.join(self.img_dir, filename)
        img = Image.open(img_path).convert('RGB') #3 colour mode (usually RGB is used thats why)

        # get all boxes for this image
        records = self.annotations[self.annotations['filename'] == filename]
        boxes = records[['xmin', 'ymin', 'xmax', 'ymax']].values.astype(np.float32) #converts into a numpy array and float32 converts the data type of that numpy array to 32 floating bit array
        boxes = torch.as_tensor(boxes, dtype=torch.float32) #tensor is basically an array but imported from pyTorch

        # all objects are "crater" → label 1
        labels = torch.ones((boxes.shape[0],), dtype=torch.int64)

        # remove invalid boxes
        valid_idx = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
        if valid_idx.sum() == 0:
            valid_boxes = torch.zeros((0, 4), dtype=torch.float32)
            valid_labels = torch.zeros((0,), dtype=torch.int64)
        else:
            valid_boxes = boxes[valid_idx]
            valid_labels = labels[valid_idx]

        # transform image
        if self.transform:
            img = self.transform(img)
        else:
            img = T.ToTensor()(img)

        target = {
            'boxes': valid_boxes,
            'labels': valid_labels,
            'image_id': torch.tensor(idx)
        }

        return img, target


#CREATE DATASET AND DATALOADER


In [13]:

transform = T.Compose([T.ToTensor()])
num_workers = 0
batch_size = 2
dataset = CraterDataset(csv_file=ANNOTATIONS_FILE, img_dir=IMAGES_DIR, transform=transform)

#OPTIONAL
img0, target0 = dataset[0]
print('Sample image shape:', img0.shape)
print('Sample target:', target0)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=lambda x: tuple(zip(*x)))

Sample image shape: torch.Size([3, 720, 720])
Sample target: {'boxes': tensor([[307., 281., 442., 403.],
        [363., 425., 431., 470.]]), 'labels': tensor([1, 1]), 'image_id': tensor(0)}


#BUILD THE MODEL

In [14]:
#USING WEIGHTS API
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 158MB/s]


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

#OPTIMIZE THE MODEL

In [15]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

#TRAINING

In [ ]:
num_epochs = 50
checkpoint_dir = '/content/drive/MyDrive/crater_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()

    if lr_scheduler is not None:
        lr_scheduler.step()

    print(f'Epoch [{epoch+1}/{num_epochs}] - Loss: {total_loss:.4f}')


final_path = os.path.join(checkpoint_dir, 'fasterrcnn_crater_final.pth')
torch.save(model.state_dict(), final_path)
print('Training complete. Saved final model to', final_path)



#SAVE FINAL MODEL

In [ ]:
final_path = os.path.join(checkpoint_dir, 'fasterrcnn_crater_final.pth')
torch.save(model.state_dict(), final_path)
print('Saved final model to', final_path)

#FASTER R-CNN TO CSV

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms

# Paths
test_dir = "/content/drive/MyDrive/LR _25/test new/test images/images"
sample_csv = "/content/drive/MyDrive/LR _25/sample submission.csv"  # uploaded sample
output_csv = "/content/drive/MyDrive/LR _25/submission.csv"

# Load sample submission to get exact filenames
sample_df = pd.read_csv(sample_csv)
filenames = sample_df["filename"].tolist()

# prepare model
model.eval()
transform = transforms.ToTensor()

results = []
row_id = 0
threshold = 0.2

for img_name in filenames:   # iterate over all expected test files
    img_path = os.path.join(test_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(img_tensor)

        boxes = outputs[0]['boxes'].cpu().numpy()
        scores = outputs[0]['scores'].cpu().numpy()

        valid_boxes = [box for box, score in zip(boxes, scores) if score >= threshold]

        if len(valid_boxes) > 0:
            x_min, y_min, x_max, y_max = map(int, valid_boxes[0])
        else:
            x_min, y_min, x_max, y_max = 0, 0, 0, 0
    else:
        # file not found → still output dummy row
        x_min, y_min, x_max, y_max = 0, 0, 0, 0

    results.append([row_id, img_name, x_min, y_min, x_max, y_max])
    row_id += 1

# Save in correct format
df = pd.DataFrame(results, columns=["row_id_column_name", "filename", "xmin", "ymin", "xmax", "ymax"])
df.to_csv(output_csv, index=False)

print(f"✅ Submission CSV saved at: {output_csv}")
print("Total rows:", len(df))


**SAVING CSV DIRECTLY IN DRIVE**

In [ ]:
df.to_csv("/content/drive/MyDrive/LR _25/submission.csv", index=False)